# **🚴‍♀️ Нелинейная модель регрессии для прогнозирования почасового спроса аренды велосипедов в Сочи в зависимости от погоды**

- Автор: Федоров Алексей Анатольевич
- Дата: 04.08.2026

<a id='цели-и-задачи-проекта'></a>
## Цели и задачи проекта

Компания BikeSouth занимается велопрокатом в южных курортных регионах. Для них характерно сильное колебание спроса на аренду в зависимости от сезона и погоды, так как летом приезжает основная масса отдыхающих, а зимой проживают преимущественно местные жители. Кроме того, спрос зависит от времени суток, а погода в пребрежных районах может сильно меняться. Поведение пользователей действительно сильно зависит от сочетания разных факторов: температуры, солнечной активности, влажности, осадков.
У заказчика уже есть линейная модель регрессии, но оказалось, что у ачсти переменных, описывающих погоду, есть нелинейные зависимости с которыми линейная модель плохо справляется и даёт неточные прогнозы. А от сочетания параметров погоды (влажность, температура, точка росы, ветер, видимость...) сильно зависит активность пользователей.

Для компании важно:
- повысить точность прогнозов;
- оптимизировать логистику распределения велосипедов;
- улучшить клиентский опыт, избегая простоев и дефицита.

Поэтому, требуется оценить метрики старой модели и построить нелинейную модель регрессии, подбирая лучшую модель на основе kNN и Decision Tree. Погодные и прочие условия в данных собираются в течении года и имеют повторяющиеся диапазоны значений, поэтому, возможность экстраполяции здесь не требуется, и kNN или DT вполне могут дать лучшие результаты.

1. **Тип задачи** — регрессия, обучение с учителем (на размеченных данных).


2. **Задача** — построение на основе наблюдений, содержащих признаки погодных условий и времени года и суток нелинейной регрессии, прогнозирующей почасовой спрос на аренду велосипедов.


3. **Цель** — оптимизация работы сервиса аренды за счёт уменьшения простоев или дефицита велосепидов, повышение удовлетворённости пользователей.


4. **Целевая переменная** — **`Rented Bike Count`**: число арендованных велосипедов в час (или, почасовой спрос на аренду).    


5. **Метрики качества модели**
    - **Основная** — **RMSE**: корень из средне-квадратичной ошибки. Сильно штрафует крупные ошибки и имеет ту же размерность, что помогает легко интерпретировать. 
$$ \mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2} $$
    - **Вспомогательные** — **R2** (объясняющая способность модели) и **MAE** (средняя абсолютная ошибка).    
    
    
6. **Требования к модели**
    - ориентировочное значение нормированной RMSE не ниже: 0.2 и ниже: $ \mathrm{nRMSE} = \frac{\mathrm{RMSE}}{y_{\max} - y_{\min}} $;
    - ориентировочное значение R2 от 0.7 и выше;
    - нужно сравнивать метрики с базовой линейной моделью, чтобы снизить RMSE.


<a id='описание-данных'></a>
## Описание данных
Данные имеются в виде датасетов:
- **ds_s14_train_data.csv** - обучающая выборка;
- **ds_s14_test_data.csv** - тестовая выборка.
Так же, имеется обученная линейная модель: **baseline_linear_regression_pipeline.joblib** (Pickle для DS).

В датасете имеются следующие признаки:
- **Temperature** — Температура воздуха в градусах Цельсия (\(°C\)). Единицы: градус Цельсия, °C.
- **Humidity** — Относительная влажность воздуха. Единицы: процент водяного пара в воздухе.
- **Windspeed** — Скорость ветра. Единицы: метр в секунду.
- **Visibility** — Видимость (в десятках метров). Единицы: десять метров.
- **Dew Point Temperature** — Точка росы (\(°C\)). Единицы: градус Цельсия, °C.
- **Solar Radiation** — Солнечная радиация (\(MJ/m^2\)). Единицы: мегаджоуль на квадратный метр, МДж/м², или MJ/m².
- **Rainfall** — Количество осадков (мм). Единицы: миллиметр, мм.
- **Snowfall** — Количество снега (см). Единицы: сантиметр, см.
- **Seasons** — Времена года: Winter, Spring, Summer, Autumn. Winter — зима, Spring — весна, Summer — лето, Autumn — осень.
- **Holiday** — Был ли в этот день выходной или праздничный день. Holiday — выходной или праздник, No Holiday — будний день.
- **Functioning Day** — Были ли велосипеды арендованы в рабочее время. Yes — да, функциональные часы, No — нет, техническое обслуживание станций проката.
- **Time_Period** — Каждая запись в датасете относится к одному из пяти временных периодов:
  - Night — с 00:00 по 05:59.
  - Morning — с 06:00 по 09:59.
  - Daytime — с 10:00 по 15:59.
  - Evening — с 16:00 по 19:59.
  - Late Evening — с 20:00 по 23:59.
  
  Для кодирования используются значения `True` и `False`.
  Данные о периоде `Daytime` — базовое состояние: если в четырёх колонках временных промежутков стоит `False`, запись относится к `Daytime`.

## Подготовка среды и библиотек

### Зависимости

Самый правильный и 100% надёжный безпроблемный вариант - использовать venv + kernel. Это даёт полную изоляцию окружения для тетради, кроме того, собственную изолированную версию Python для venv, которую умеет ставить Conda:
```
conda create -y -n "${ENV_NAME}" python="${PYTHON_VERSION}"
```
Однако, ревьюверы почему-то не любят переключать ядра Jupyter, поэтому, будем использовать установку библиотек в домашнюю папку под Python 3.9. При этом не исключены конфликты версий.

**1.** Обновляем pip, чтобы работал --dry-run.

In [1]:
%pip install --upgrade --user --disable-pip-version-check pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------- ----- 1.6/1.8 MB 20.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 15.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
%pip -V

pip 26.2.1 from C:\Users\user\AppData\Roaming\Python\Python314\site-packages\pip (python 3.14)

Note: you may need to restart the kernel to use updated packages.


**2.** Проверяем наличие scikit-learn нужной версии через --dry-run.

In [3]:
%pip install --dry-run --only-binary=:all: --disable-pip-version-check scikit-learn==1.6.1

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scikit-learn==1.6.1 (from versions: 1.7.2, 1.8.0rc1, 1.8.0, 1.9.0rc1, 1.9.0)
ERROR: No matching distribution found for scikit-learn==1.6.1


Тут ставим зависимости. --user - ставит в домашнюю папку пользователя /home/xxx_user_name/.local/lib, а не в /opt/conda/lib. В sys.path прописана сначала папка пользователя, поэтому модули возьмутся оттуда.

--only-binary - не используется компиляция, а только Wheels - готовые бинарники.

In [4]:
"""
%pip install --user --only-binary=:all: --disable-pip-version-check \
    numpy==1.26.4 \
    scipy==1.13.1 \
    pandas==2.1.4 \
    scikit-learn==1.6.1 \
    matplotlib==3.8.2 \
    seaborn==0.13.0 \
    statsmodels==0.14.1 \
    phik==0.12.5 \
    jinja2==3.1.4 \
    numba==0.58.1 \
    llvmlite==0.41.1 \
    patsy==0.5.6 \
    joblib==1.4.2 \
    threadpoolctl==3.5.0
""";

In [5]:
%pip install --user --only-binary=:all: --disable-pip-version-check --no-warn-conflicts \
    scikit-learn==0.24.1 \
    numpy==1.21.6 \
    scipy==1.7.3 \
    pandas==1.3.5 \
    joblib==1.1.1 \
    threadpoolctl==3.1.0 \
    matplotlib==3.5.3 \
    seaborn==0.12.2 \
    statsmodels==0.13.5 \
    patsy==0.5.6 \
    phik==0.12.2 \
    numba==0.56.4 \
    llvmlite==0.39.1 \
    jinja2==3.1.4

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scikit-learn==0.24.1 (from versions: 1.7.2, 1.8.0rc1, 1.8.0, 1.9.0rc1, 1.9.0)
ERROR: No matching distribution found for scikit-learn==0.24.1


**3.** Проверяем версии библиотек.

👉 **Нужен Restart Kernel !!!**

In [6]:
import sys, os
import numpy, scipy, pandas, sklearn, matplotlib
import seaborn, statsmodels, phik, numba, llvmlite

print(f"python      : {sys.version.split()[0]}")
for m in (numpy, scipy, pandas, sklearn, matplotlib,
          seaborn, statsmodels, phik, numba, llvmlite):
    print(f"{m.__name__:<12}: {m.__version__:<10} <- {os.path.dirname(m.__file__)}")

# assert sklearn.__version__ == "1.6.1", f"sklearn = {sklearn.__version__}, ожидался 1.6.1"
# assert numpy.__version__ == "1.26.4", f"numpy = {numpy.__version__}, ожидался 1.26.4"
# print("\n✅ Всё встало корректно, sklearn 1.6.1 на Python 3.9")

assert sklearn.__version__ == "0.24.1", f"sklearn = {sklearn.__version__}, ожидался 0.24.1"
assert numpy.__version__ == "1.21.6", f"numpy = {numpy.__version__}, ожидался 1.26.6"
print("\n✅ Всё встало корректно, sklearn 0.24.1 на Python 3.9")

ModuleNotFoundError: No module named 'numpy'

👉 Если тут ошибка, то **Нужен Restart Kernel !!!**

**4.** Удаление (очистка) установленных модулей.
Этот код нужно раскомментить и выполнить после проверки работы или в случае проблем с установкой, если был непустой `/home/xxx_user_name/.local/lib/python3.9/site-packages/`

In [ ]:
# %pip uninstall -y numpy scipy pandas scikit-learn matplotlib seaborn statsmodels phik numba llvmlite

In [ ]:
# import shutil
# shutil.rmtree("/home/jovyan/.local/lib/python3.9/site-packages", ignore_errors=True)

### Импорты библиотек

In [ ]:
# тут сделаем импорты снова, не учитывая технологические импорты для установки, чтобы была единая точка для удобства
import contextlib
import gc
import os
import sys
import time
import warnings
import inspect
from pprint import pprint
from datetime import datetime
from dataclasses import dataclass
from functools import reduce
from operator import mul
from itertools import combinations
from typing import Protocol, Callable

import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import phik
import seaborn as sns
from IPython.display import HTML, Markdown, display_html
import joblib
from joblib import Memory
from matplotlib.ticker import PercentFormatter
from phik import phik_matrix

import sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

In [ ]:
print(
    f"Путь к Python: {sys.executable}",
    f"Версия Python: {sys.version}",
    f"Текущий путь: {os.getcwd()}",
    f"pandas: {pd.__version__}",
    f"numpy: {np.__version__}"
    f"\n\tБэкенд pandas (plotting): {pd.options.plotting.backend}",
    f"phik: {phik.__version__}",
    f"matplotlib: {matplotlib.__version__}",
    f"seaborn: {sns.__version__}",
    f"sklearn: {sklearn.__version__}",
sep='\n')

<a id='общие-функции'></a>
### Общие функции

<a id='базовые'></a>
#### Базовые

In [ ]:
def to_snake_case_columns(dataframe: pd.DataFrame) -> None:
    """Приводит названия колонок к snake_case прямо в исходном датафрейме."""
    dataframe.columns = (
        dataframe.columns
        .str.replace(r'[\s/()\-%,.]+', '_', regex=True)  # Замена спецсимволов и пробелов на _
        .str.replace(r'^_+|_+$', '', regex=True)         # Удаление подчеркиваний на концах
        .str.lower()                                     # Перевод в нижний регистр
    )
    
    
def get_integer_features_info(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Находит признаки, содержащие только целые числа (игнорируя NaN/пропуски),
    и возвращает DataFrame с их именами и кардинальностью.
    """
    integer_features = []
    cardinalities = []
    
    for col in dataframe.columns:
        # Удаляем пропуски для корректной проверки значений
        clean_col = dataframe[col].dropna()
        
        # Если колонка стала пустой после удаления NaN, пропускаем её
        if clean_col.empty:
            continue
            
        # Проверяем, является ли колонка числовой
        if pd.api.types.is_numeric_dtype(clean_col):
            # Проверяем, что все значения являются целыми (остаток от деления на 1 равен 0)
            if (clean_col % 1 == 0).all():
                integer_features.append(col)
                cardinalities.append(clean_col.nunique())
        else:
            # Если тип object/string, пробуем привести к числу
            try:
                converted = pd.to_numeric(clean_col)
                if (converted % 1 == 0).all():
                    integer_features.append(col)
                    cardinalities.append(converted.nunique())
            except (ValueError, TypeError):
                continue
                
    # Формируем итоговый DataFrame
    result_df = pd.DataFrame({
        'feature_name': integer_features,
        'cardinality': cardinalities
    })
    
    return result_df
    
def downcast_columns_for_ml(data_frame, *, ints=None, floats=None, cats=None):
    """Приводит колонки к оптимальным NumPy-типам для ML с downcast.

    - Integer колонки: int64 -> int8/16/32/64 в зависимости от диапазона.
    - Float колонки: float64 -> float32.
    - Object/String колонки: object -> category (для экономии памяти и ML).

    ВАЖНО: В отличие от Extended-типов pandas, стандартные целочисленные типы
    NumPy (int8-int64) НЕ поддерживают NaN. Если в целочисленной колонке есть
    пропуски, pandas принудительно оставит её в типе float32/64.
    """

    ints_actual = (
        set(ints) if ints is not None
        else set(data_frame.select_dtypes(include="integer").columns)
    )
        
    floats_actual = (
        set(floats) if floats is not None
        else set(data_frame.select_dtypes(include="float").columns)
    ) - ints_actual    
    
    cats_actual = (
        set(cats) if cats is not None
        else set(data_frame.select_dtypes(include=["object", "string"]).columns)
    )

    for col in ints_actual:
        # Приводим к числовому типу NumPy
        data_frame[col] = pd.to_numeric(data_frame[col], errors="coerce")

        # Если в колонке есть NaN, NumPy int не сможет её сохранить.
        # В таком случае переводим её во float32, чтобы не ломать ML-алгоритмы.
        if data_frame[col].isnull().any():
            data_frame[col] = data_frame[col].astype(np.float32)
            continue

        col_min, col_max = data_frame[col].min(), data_frame[col].max()

        if col_min >= -128 and col_max <= 127:
            data_frame[col] = data_frame[col].astype(np.int8)
        elif col_min >= -32768 and col_max <= 32767:
            data_frame[col] = data_frame[col].astype(np.int16)
        elif col_min >= -2147483648 and col_max <= 2147483647:
            data_frame[col] = data_frame[col].astype(np.int32)
        else:
            data_frame[col] = data_frame[col].astype(np.int64)

    for col in floats_actual:
        data_frame[col] = pd.to_numeric(data_frame[col], errors="coerce")
        data_frame[col] = data_frame[col].astype(np.float32)

    # Для ML текстовые признаки переводим в категориальный тип, а не в string
    for col in cats_actual:
        data_frame[col] = data_frame[col].astype("category")

    return data_frame

<a id='базовый-eda'></a>
#### Базовый EDA

In [ ]:
def normalize_text_cols(df, col_names, to_lower=True):
    """
        Конвертирует колонки col_names в string, обрезает пробелы с концов и нормализует пробелы внутри строк.
        Если задано to_lower, то приводит к нижнему регистру.
    """
    for name in col_names:        
        df[name] = df[name].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
        if to_lower:
            df[name] = df[name].str.lower()

def get_missing_rows_info(dataframe: pd.DataFrame) -> dict:
    """Возвращает количество и процент строк, содержащих хотя бы один пропуск."""
    missing_mask = dataframe.isna().any(axis=1)
    
    total_missing = missing_mask.sum()
    percent_missing = (missing_mask.mean() * 100).round(2)
    
    return {
        'missing_rows_count': total_missing,
        'missing_rows_percent': percent_missing
    }
            
def get_whiskers(values, coeff=1.5, left=None, right=None):
    """
        Возвращает усы ящика для серии values.
    """
    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)
    IQR = Q3 - Q1
    
    w_left = Q1 - coeff * IQR
    w_right = Q3 + coeff * IQR
    
    if left is not None:
        w_left = max(left, w_left)
    if right is not None:
        w_right = min(right, w_right)
        
    return w_left, w_right            
            
def get_reference_info(df, name, numeric=False, limit = 50):
    """
        Выполняет проверку серии name из DataFrame df как категории. А именно:
            - выводит TOP 50 уникальных значений серии и общее число уникальных значений;
            - выводит счётчик встречаемости и % для каждого значения в серии name;
            - если серия числовая, то ещё выводит min и max значения.
        Таким образом, помогает оценить качество справочника или просто серии (колонки) и заметить неявные дубликаты.
    """
    values = df[name].drop_duplicates().sort_values()
    len_values = len(values)
    lst = values.head(limit).to_list()
    
    elipsis = ', ...' if len_values > limit else ''
    display(Markdown(f"**{name}:** `{lst}{elipsis}` = **{len_values}**\n\n"))
        
    counts_df = df[name].value_counts(dropna=False).astype('float64').to_frame('count')
    counts_df.index = counts_df.index.map(lambda x: '<NA>' if pd.isna(x) else x)
    counts_df['%'] = (counts_df['count'] / counts_df['count'].sum() * 100).round(2)        
    counts_df['count'] = counts_df['count'].astype('Int64')
             
    display(counts_df)
            
    print('\nПропущено:', df[name].isna().sum())
    
    if numeric:
        values_num = pd.to_numeric(df[name], errors='coerce')        
        min_val = values_num.min()
        max_val = values_num.max()
        print(f'min = {min_val}, mean = {round(values_num.mean(), 2)}, max = {max_val}, median = {values_num.median()}')
                
        whisker_left, whisker_right = get_whiskers(values_num, left=min_val, right=max_val)
        print(f'left_whisker = {whisker_left}, right_whisker = {whisker_right}')

        non_numeric = values[values_num.isna()].tolist()
        if non_numeric:
            print(f'Нецифровые значения: {non_numeric}')
    print('\n')
    
def get_frequency_report(df, name):
    """
        Возвращает частотный отчёт по колонке данных: все характеристики повторяемости значений в разных строках.
    """
    counts = df[name].value_counts(dropna=False)
    return pd.Series({
        'Среднее': counts.mean(),
        'Медиана': counts.median(), 
        'Мода': counts.mode().iloc[0] if len(counts.mode()) > 0 else None,
        'Макс': counts.max(),
        'Мин': counts.min()
    }), counts

def get_quasi_consts_numeric(df, names, threshold=0.1):
    vs_quasi = VarianceThreshold(threshold=threshold)     
    numerical_data = df[list(names)]
    vs_quasi.fit(numerical_data)
    return [col for col, keep in zip(names, vs_quasi.get_support()) if not keep]

def get_quasi_consts_categories(df, names):
    return [col for col in list(names)
                      if df[col].value_counts(normalize=True).iloc[0] > 0.90]

def drop_columns_inplace_safe(df, cols):
    existing_cols = df.columns.intersection(cols)
    df.drop(columns=existing_cols, inplace=True)   
    
def str_to_binary(dataframe: pd.DataFrame, column: str, mapping: dict) -> None:
    """
    Преобразует бинарный текстовый признак в число на месте (inplace).
    mapping: {'holiday': 1, 'no holiday': 0}
    Значения вне mapping и пропуски -> NaN.
    dtype: int8, если пропусков нет, иначе float32 (совместимо с sklearn).
    """
    s = dataframe[column].map(mapping)
    dataframe[column] = s.astype('int8') if s.notna().all() else s.astype('float32')

#### Статистический анализ

In [ ]:
def check_vif(df, fields, top=10):
    """Расчет фактора инфляции дисперсии (VIF) для проверки мультиколлинеарности.

    Интерпретация значений VIF:
    - VIF = 1: коллинеарность отсутствует.
    - 1 < VIF <= 5: умеренная коллинеарность (допустимо).
    - VIF > 10: критический уровень, признак рекомендуется удалить.

    Важно: функция удаляет строки с пропусками (dropna).
    """
    df_vif = df[list(fields)].astype(float).dropna()

    df_with_const = add_constant(df_vif)

    vif_data = pd.DataFrame()
    vif_data["Feature"] = df_vif.columns

    vif_data["VIF"] = [
        variance_inflation_factor(
            df_with_const.values, df_with_const.columns.get_loc(col)
        )
        for col in df_vif.columns
    ]

    return vif_data.sort_values(by="VIF", ascending=False).head(top)

def print_high_corr_spearman_pairs(corr, threshold=0.7):
    print("\nСИЛЬНО СКОРРЕЛИРОВАННЫЕ ПАРЫ (Spearman |rho| > 0.7):")
    print("-" * 60)

    pairs = corr.unstack().reset_index()
    pairs.columns = ["Feature 1", "Feature 2", "Spearman"]

    pairs = pairs[pairs["Feature 1"] < pairs["Feature 2"]]

    high_corr_pairs = pairs[pairs["Spearman"].abs() > threshold].sort_values(
        by="Spearman",
        key=lambda s: s.abs(),
        ascending=False
    )

    if not high_corr_pairs.empty:
        for _, row in high_corr_pairs.iterrows():
            print(f"{row['Feature 1']:20} <---> {row['Feature 2']:20} | Spearman: {row['Spearman']:.4f}")
    else:
        print(f"✅ Сильно скоррелированных пар с |Spearman| > {threshold} не обнаружено.")

    print("-" * 60)

    return high_corr_pairs

#### Графики

In [ ]:
def plot_continuous_distributions(data_df, vars_list, kind="boxen", log_scale=False, bins="auto"):
    """
    Распределение непрерывных признаков: одна строка — одна переменная.
    Слева гистограмма с KDE, справа boxen/box на той же шкале X.

    kind      : 'boxen' (тяжёлые хвосты) | 'box' | 'violin'
    log_scale : True -> лог-шкала X (для правоскошенных величин)
    """
    nrows = len(vars_list)
    fig, axes = plt.subplots(nrows, 2, figsize=(14, 4.8 * nrows), squeeze=False,
                             gridspec_kw={"width_ratios": [1.6, 1]})

    for i, var in enumerate(vars_list):
        s = data_df[var].dropna()
        ax_l, ax_r = axes[i, 0], axes[i, 1]

        sns.histplot(x=s, bins=bins, kde=True, color="#4C72B0",
                     edgecolor="white", linewidth=0.3, ax=ax_l)
        ax_l.axvline(s.mean(), color="#C44E52", ls="--", lw=1.5, label=f"mean {s.mean():.1f}")
        ax_l.axvline(s.median(), color="#55A868", ls="-", lw=1.5, label=f"median {s.median():.1f}")
        ax_l.legend(fontsize=9)
        ax_l.set_title(f"Распределение: {var}", fontsize=13, pad=10, fontweight='bold')
        ax_l.set_ylabel("Count", fontsize=11)

        plot = {"boxen": sns.boxenplot, "box": sns.boxplot, "violin": sns.violinplot}[kind]
        plot(x=s, color="#4C72B0", ax=ax_r, **({"width": 0.4} if kind != "boxen" else {}))
        ax_r.scatter(s.mean(), 0, marker="D", s=45, color="#C44E52", zorder=10, label="mean")
        ax_r.legend(fontsize=9, loc="upper right")

        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        n_out = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
        ax_r.set_title(f"{kind}: {var} | выбросов по IQR: {n_out} ({n_out / len(s):.1%})",
                       fontsize=13, pad=10, fontweight='bold')
        ax_r.set_xlabel(var, fontsize=11)

        if log_scale:
            for ax in (ax_l, ax_r):
                ax.set_xscale("symlog")   # symlog, а не log: переносит нули

        stats = (f"n={len(s)}  zeros={(s == 0).sum()}  "
                 f"std={s.std():.1f}  skew={s.skew():.2f}  kurt={s.kurtosis():.2f}  "
                 f"min={s.min():.1f}  max={s.max():.1f}")
        ax_l.annotate(stats, xy=(0.5, -0.3), xycoords="axes fraction",
                      ha="center", fontsize=12, color="#555")

    plt.tight_layout()
    plt.show()

#### Обучение

In [ ]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Совместимый с новыми joblib патч через print_progress."""
    original_print_progress = joblib.parallel.Parallel.print_progress

    def patched_print_progress(self):
        # n_completed_tasks обновляется joblib'ом
        tqdm_object.n = self.n_completed_tasks
        tqdm_object.refresh()
        return original_print_progress(self)

    joblib.parallel.Parallel.print_progress = patched_print_progress
    try:
        yield tqdm_object
    finally:
        joblib.parallel.Parallel.print_progress = original_print_progress
        tqdm_object.close()
        
def sigmoid(x):
    return 1 / (1  + np.exp(-x))

##### Настройки библиотек

In [ ]:
memory = Memory(location='./cache_pipeline', verbose=0)
sns.set_theme(style="whitegrid") # светлосерая сетка

In [ ]:
LOCAL_DATA_PATH = '/datasets/'
S3_DATA_PATH = 'https://code.s3.yandex.net/datasets/'
SEED = 42
TARGET = 'rented_bike_count'

class BikeSouthProto(Protocol):
    train_set: str
    test_set: str
    linear_model: str    

@dataclass(frozen = True)
class BikeSouthFiles:
    train_set: str = 'ds_s14_train_data.csv'
    test_set: str = 'ds_s14_test_data.csv'
    linear_model: str = 'baseline_linear_regression_pipeline.joblib'

## Загрузка данных, ознакомление и начальная обработка

#### Общий код

In [ ]:
def load_df(file_name):
    local_path = LOCAL_DATA_PATH + file_name
    s3_path = S3_DATA_PATH + file_name
    
    data_path = local_path if os.path.exists(local_path) else s3_path
    df = pd.read_csv(data_path, decimal='.', sep=',', skipinitialspace=True)
    df_shape = df.shape
    print(f'✅ Датасет загружен с "{data_path}" содержит: {df_shape[0]} строк')    
    
    return df

def pre_analyze_ds(df):    
    display(df.dtypes.to_frame(name='type'))
    display(df.head())
    
    print('Явные полные дубликаты:', df.duplicated().sum())
    
    missing_df = df.isna().sum().to_frame(name='missing_count')
    missing_df['missing_percent'] = (df.isna().mean() * 100).round(2)
    display(missing_df)

#### Обучающая выборка

In [ ]:
df_train = load_df(BikeSouthFiles.train_set)
df_train.attrs['name'] = BikeSouthFiles.train_set

In [ ]:
# сохраняем копию для сравнения (если понадобится) и для проверки линейной baseline модели
df_train_source = df_train.copy()

In [ ]:
normalize_text_cols(df_train, df_train.select_dtypes(include=['object']).columns)

✅ Нормализовали текстовые поля и преобразовали в string.

👉 Добавить в pipeline.

In [ ]:
get_missing_rows_info(df_train)

In [ ]:
pre_analyze_ds(df_train)

In [ ]:
df_train.info(True, None, None, True)

In [ ]:
to_snake_case_columns(df_train)
print(df_train.columns.tolist())

✅ Привели имена колонок к Snake Case.

👉 Добавить в pipeline.

Теперь можно определить типы признаков. Для этого проверим типы и кардинальность признаков.

Смотрим, какие из числовых признаков целые:

In [ ]:
display(get_integer_features_info(df_train))

Проверяем, что у нас находится в категориальных признаках:

In [ ]:
get_reference_info(df_train, 'holiday')

In [ ]:
str_to_binary(df_train, 'holiday', {'holiday': 1, 'no holiday': 0})

✅ Привели holiday к binary.

👉 Добавить в pipeline.

In [ ]:
get_reference_info(df_train, 'functioning_day')

In [ ]:
str_to_binary(df_train, 'functioning_day', {'yes': 1, 'no': 0})

✅ Привели functioning_day к binary.

👉 Добавить в pipeline.

In [ ]:
get_reference_info(df_train, 'seasons')

In [ ]:
BIN_VARS = (
    'holiday', 'functioning_day', 'time_period_evening', 
    'time_period_late_evening', 'time_period_morning', 'time_period_night', 
    'rented_bike_count',)

CAT_VARS = ('seasons', )

CONTINIOUS_INT_VARS = ('visibility_10m', 'rainfall_mm', 'snowfall_cm')

CONTINIOUS_REAL_VARS = (
    'temperature', 'humidity', 'wind_speed_m_s', 
    'dew_point_temperature', 'solar_radiation_mj_m2',)

Проверяем все ли признаки описаны:

In [ ]:
set((BIN_VARS) + (CAT_VARS) + (CONTINIOUS_INT_VARS) + (CONTINIOUS_REAL_VARS) + (TARGET,)) - \
set(df_train.columns.tolist())

✅ На основании анализа значений признаков распредилили их по типам для раздельной дальнейшей обработки.

In [ ]:
downcast_columns_for_ml(
    df_train, 
    ints=CONTINIOUS_INT_VARS + (TARGET,) + BIN_VARS, 
    cats=CAT_VARS, 
    floats=CONTINIOUS_REAL_VARS
);

SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

df_train["seasons"] = df_train["seasons"].cat.reorder_categories(
    SEASON_ORDER,
    ordered=True
)

✅ Оптимизировали типы признаков датасета.

👉 Добавить в pre_pipeline.

Проверяем, что получилось:

In [ ]:
df_train.info(True, None, None, True)

👉 Потребление памяти снизилось с 684.5 KB до 280.9 KB: в 2.4 раза.

In [ ]:
pre_analyze_ds(df_train)

#### Тестовая выборка
Тут только убедимся в её целостности и соотвествии обучающей выборке по колонкам и типам. Преобразования не нужны, так как она будет использоваться через pipeline и не будет использоваться в анализе.

In [ ]:
df_test = load_df(BikeSouthFiles.test_set)
df_test.attrs['name'] = BikeSouthFiles.test_set

In [ ]:
# проверяем совпадение колонок
print(df_test.columns.equals(df_train_source.columns))
print(set(df_train_source.columns) ^ set(df_test.columns))

In [ ]:
get_missing_rows_info(df_test)

In [ ]:
pre_analyze_ds(df_test)

👉 Тестовая выборка совпадает по структуре с обучающей и имеет почти столько же строк с пропусками: 19%. В анализе эту выборку не используем, поэтому дальнейшая обработка не нужна. Выборка пригодится в исходном виде для тестирования пайплайна и оценки модели.

### Промежуточный вывод

**Итоги по данным**

- **Обучающая выборка:** 7 008 строк, 16 признаков.
- **Тестовая выборка:** 1 752 строки, полная структурная идентичность обучающей.
- **Целевая переменная:** `rented_bike_count` (`int`), распределение требует проверки на выбросы.
- **Пропуски:** ~19% строк содержат пропуски, в основном в погодных признаках. Пропуски не случайны — коррелируют с погодными условиями, поэтому требуется определить стратегию обработки.
- **Дубликаты:** отсутствуют.

**Предварительные преобразования**

- **Нормализация текста:** все строковые колонки приведены к нижнему регистру, удалены лишние пробелы.
- **Snake case:** имена колонок приведены к единому стилю.
- **Бинаризация:** `holiday`, `functioning_day` переведены в 0/1.
- **Типизация:**
  - категориальный признак `seasons` (4 сезона) оставлен как категория — требуется one-hot кодирование в пайплайне;
  - непрерывные признаки разделены на целочисленные (`visibility_10m`, `rainfall_mm`, `snowfall_cm`) и вещественные — это позволит применять разные стратегии анализа и обработки;
  - битовые флаги времени суток (`time_period_*`) оставлены как бинарные;
  - потребление памяти снизилось в 2.4 раза.
- **Оптимизация памяти:** все колонки приведены к минимальныс типам (`float32`, `int8`, `int16`) — экономия памяти ~40% без потери точности.

**Проблемные зоны, требующие внимания**

- **Пропуски в погодных признаках** (до 3.7%): требуется стратегия заполнения.
- **Нельзя удалять строки:** потеряем 19% данных, что критично для временных рядов.
- **Высокая кардинальность `visibility_10m`** (1697 уникальных значений): возможен эффект шума, можно рассмотреть биннинг или логарифмическое преобразование.

## Анализ старой линейной модели

### Загрузка модели

In [ ]:
local_model_path = LOCAL_DATA_PATH + BikeSouthFiles.linear_model
s3_model_path = S3_DATA_PATH + BikeSouthFiles.linear_model

model_path = local_model_path if os.path.exists(local_model_path) else s3_model_path

model_info = joblib.load(model_path)

✅ Загрузили модель (требует scikit-learn 0.24.1).

In [ ]:
pipe = model_info["model"] if isinstance(model_info, dict) else model_info
print(pipe) 

In [ ]:
# проверяем, что достали
pipe.named_steps.keys()

### Тестирование качества модели

In [ ]:
X = df_test.drop(columns=["Rented Bike Count"])
y = df_test["Rented Bike Count"]

pred = pipe.predict(X)

In [ ]:
# аренда не бывает отрицательной, но линейные модели экстраполируют и ночью могут уводить прогноз в минус
pred_clipped = np.clip(pred, 0, None)
# настоящий baseline (всегда среднее начение TARGET)
pred_dummy = np.full(len(y), y.mean())


def score(y_true, y_pred, name):
    return {
        "модель": name,
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "WAPE": mean_absolute_error(y_true, y_pred) / y.mean(),
    }


base_metrics = pd.DataFrame([
    score(y, pred, "linear (как есть)"),
    score(y, pred_clipped, "linear + clip(0)"),
    score(y, pred_dummy, "dummy (mean)"),
]).set_index("модель")

display(base_metrics.style.format({"RMSE": "{:.2f}", "MAE": "{:.2f}", "R2": "{:.4f}"})
                     .highlight_min(subset=["RMSE", "MAE"], color="#d4f8d4")
                     .highlight_max(subset=["R2"], color="#d4f8d4"))

**Отрицательные прогнозы:**

In [ ]:
diag = pd.Series({
    "прогнозов < 0": (pred < 0).sum(),
    "доля < 0, %": round((pred < 0).mean() * 100, 1),
    "min pred": round(pred.min(), 1),
    "max pred": round(pred.max(), 1),
    "y mean": round(y.mean(), 1),
    "y std": round(y.std(), 1),
    "y max": y.max(),
    "n": len(y),
}, name="значение").to_frame()

display(diag)

### Важность признаков по данной модели

In [ ]:
r = permutation_importance(pipe, X, y, n_repeats=10, random_state=SEED,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)

print(pd.DataFrame({"rmse_drop": r.importances_mean, "std": r.importances_std},
                   index=X.columns).sort_values("rmse_drop", ascending=False).round(2))

### Промежуточный вывод
Линейная модель объясняет **60%** дисперсии целевой переменной, что значительно выше dummy. Это средний показатель.

**RMSE** упало относительно Dummy на **37%**. Это существенное улучшение и значит, что модель полезна.

**MAE** составляет **43%** от среднего спроса в **700** велосипедов. Для планирования такая ошибка велика.

**RMSE / MAE = 1.36** - есть крупные ошибки. **Max pred = 1964, а max = 3380**. Это значит, что модель сглаживает пики, не дотягивается. И там большая ошибка.

**8.4%** отрицательных предсказаний. Это может быть в ночные периоды и в дни, когда прокат закрыт. Вызвано экстраполяцией линейной модели. B clip(0) даёт снижение MAE.

Типичный спрос на прокат имеет суточные пики, вероятно замена признака Hour на категорийный исказила распределение.
Ещё, зависимость от темпиратуры и от дождя скорее всего не монотонна, и это плохо учитывает текущая линейная модель.

## Исследовательский анализ данных

### Целевая переменная rented_bike_count

In [ ]:
df_train[TARGET].describe()

In [ ]:
df_train[TARGET].mode().iloc[0]

In [ ]:
plot_continuous_distributions(df_train, [TARGET])

Пропусков нет, как видели при первичном анализе.

Среднее сильно выше медианы и skewness = 1.16 - это правосторонняя ассиметрия распределения. Std = 646 очень близко в mean. Это значит, что очень большой разброс спроса. Спрос очень меняется. 

Много нулей: 242. Это видимо нерабочие периоды. Выбросов 1.7%, они в правом хвосте. Пиковые значения вероятно соотвествуют редким праздникам и выходным дням с хорошей погодой.

Максимальная плотность распределения 0 - 1500. Оно похоже на гамма-распределение. 

Куртиозис 0.87 показывает средне-тяжёлый хвост.

Мода - 0. Самое частое значение - аренды нет вообще, а наиболее частый диапазон спроса 0 - 250. Здесь максимальная плотность распределения. 

Это похоже на Zero-inflated процесс. То есть, нулей много и они не случайны.

👉 Для случая **Zero-Inflated** надо разбивать целевую переменную на две и строить две модели (для вероятности класса нуль / не нуль и для отдельной регрессии на выборке без нулей:

    - y_clf = (rented_bike_count > 0).astype(int) — для классификатора;
    - y_reg = np.log1p(rented_bike_count[rented_bike_count > 0]) — для регрессии.

Но поскольку у нас не очень много нулевых периодов (3.4%), а модели kNN и Decision Tree, то ограничимся только логарифмированием через log1p.

👉 Добавить логарифмирование **log1p** TARGET на обоих выборках. А в обёртке вызвать **np.expm1**, чтобы получить число велосипедов.

### Категориальные переменные

In [ ]:
# CAT_VARS = ('seasons', )

In [ ]:
def plot_feature_analysis(df, target, feature):
    """
    Визуализирует распределение целевой переменной по категориям признака.
    Порядок категорий автоматически берется из типа данных Categorical.
    """
    # Автоматическое извлечение порядка из категориального признака
    feature_order = list(df[feature].cat.categories)
    num_categories = len(feature_order)

    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.3, 1], hspace=0.3, wspace=0.3)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[1, :])

    # 1. Boxenplot
    sns.boxenplot(data=df, x=feature, y=target, order=feature_order,
                  palette="Set2", ax=ax0)
    means = df.groupby(feature)[target].mean().reindex(feature_order)
    ax0.scatter(range(num_categories), means, marker="D", s=60, color="#C44E52", zorder=10, label="mean")
    ax0.legend()
    ax0.set_title(f"Распределение спроса по {feature}", fontweight='bold', pad=15)

    # 2. KDE
    sns.kdeplot(data=df, x=target, hue=feature, hue_order=feature_order,
                common_norm=False, fill=True, alpha=0.25, palette="Set2", ax=ax1)
    ax1.set_title(f"KDE по {feature} (common_norm=False)", fontweight='bold', pad=15)

    # 3. ECDF
    sns.ecdfplot(data=df, x=target, hue=feature, hue_order=feature_order,
                 palette="Set2", lw=2, ax=ax2)
    ax2.set_title("ECDF: доля часов со спросом ≤ X", fontweight='bold', pad=10)
    ax2.grid(True, linestyle='--', alpha=0.3)

    # Общий заголовок и подгонка отступов
    fig.suptitle(f"Анализ {target} по {feature}", fontsize=16, fontweight='bold', y=1.0)
    plt.subplots_adjust(top=0.92)
    
    plt.show()

In [ ]:
plot_feature_analysis(df_train, TARGET, 'seasons')

## Финальная очистка

In [ ]:
memory.clear(warn=False)
print("✅ Кэш препроцессинга очищен.")